# Chapter 33 — Sequence Models and the Arrival of Attention

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then `docs/TROUBLESHOOTING.md`.

In [1]:
!pip -q install -r https://raw.githubusercontent.com/FromAbsoluteZero/CodeBase/main/requirements.txt  # Colab only; skip locally

zsh:1: command not found: pip


## Shared setup

Imports and the objects the blocks below reuse. The chapter prints these once and then continues the same session. This cell is `code/ch33/_lib.py`.

In [2]:
import numpy as np, warnings; warnings.filterwarnings("ignore")
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

digits = load_digits()
X_img = digits.images / 16.0            # (n, 8, 8): 8 rows, read as 8 timesteps
X, y = digits.data / 16.0, digits.target
Xtr_img, Xte_img, ytr, yte = train_test_split(X_img, y, test_size=0.2,
                                              stratify=y, random_state=0)
Xtr, Xte = Xtr_img.reshape(len(Xtr_img), -1), Xte_img.reshape(len(Xte_img), -1)

def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

## The chapter code

### Block 1  (`c1.py`)

In [3]:
# A recurrent network reads a sequence one step at a time, carrying a
# hidden state forward. The same weights are reused at every step: this
# is parameter sharing across time, exactly as Chapter 32 shared a
# filter's weights across space.
r = np.random.default_rng(33)
D_in, D_hid = 8, 16                  # 8 pixels per row, 16-unit hidden state
Wx = r.normal(0, np.sqrt(1/D_in), (D_in, D_hid))
Wh = r.normal(0, np.sqrt(1/D_hid), (D_hid, D_hid))
bh = np.zeros(D_hid)

def rnn_step(x_t, h_prev):
    return np.tanh(x_t @ Wx + h_prev @ Wh + bh)

img = Xtr_img[0]                         # one digit, 8 rows
h = np.zeros(D_hid)                      # hidden state starts at zero

print(f"{'timestep':>9}{'row (input)':>14}{'|hidden state|':>16}")
for t in range(8):
    h = rnn_step(img[t], h)
    print(f"{t:>9}{str(img[t].shape):>14}{np.linalg.norm(h):>16.4f}")

print(f"\nafter all 8 rows, one hidden state of size {D_hid} summarizes")
print(f"the entire image:")
print(h.round(3))

 timestep   row (input)  |hidden state|
        0          (8,)          2.1026
        1          (8,)          2.2817
        2          (8,)          2.0675
        3          (8,)          2.0757
        4          (8,)          2.1791
        5          (8,)          2.1717
        6          (8,)          2.4250
        7          (8,)          2.6378

after all 8 rows, one hidden state of size 16 summarizes
the entire image:
[-0.579  0.809  0.798 -0.067 -0.242 -0.004 -0.738  0.509 -0.034  0.937
 -0.882  0.746 -0.898 -0.808  0.779 -0.435]


### Block 2  (`c2.py`)

In [4]:
# The same vanishing-gradient problem from Chapter 31, on a new axis:
# instead of shrinking through many LAYERS, the gradient here shrinks
# through many TIME STEPS, because the same weight matrix multiplies
# the signal at every step, exactly like Chapter 31's Wh at every layer.
def tanh_grad(h):
    return 1 - h**2

r2 = np.random.default_rng(33)
D_hid = 16
# deliberately small-scale recurrent weights
Wh_decay = r2.normal(0, 0.3, (D_hid, D_hid))

def run_and_track_grad(n_steps, Wh_):
    h = np.zeros(D_hid)
    hs = [h]
    x = r2.normal(size=(n_steps, D_hid))
    for t in range(n_steps):
        h = np.tanh(x[t] + h @ Wh_)
        hs.append(h)
    # gradient of the LAST hidden state w.r.t. the FIRST: one factor per
    # step, Wh then tanh'(h_t), in forward order (they do not commute)
    grad = np.eye(D_hid)
    for t in range(1, n_steps + 1):
        grad = grad @ (Wh_ @ np.diag(tanh_grad(hs[t])))
    return np.linalg.norm(grad)

print(f"{'sequence length':>16}{'gradient norm, step 1 to last':>30}")
for n in (5, 10, 20, 40, 80):
    g = run_and_track_grad(n, Wh_decay)
    print(f"{n:>16}{g:>30.2e}")

 sequence length gradient norm, step 1 to last
               5                      5.15e-01
              10                      1.48e-01
              20                      1.02e-03
              40                      3.97e-06
              80                      5.66e-15


### Block 3  (`c3.py`)

In [5]:
# Backpropagation through time: the chain rule applied along the time
# axis, exactly as Chapter 32 applied it along space. Each step's
# gradient must account for two paths forward: into the output at that
# step, and into the next hidden state.
def rnn_forward(imgs, Wx, Wh, bh):
    n, T, D_in = imgs.shape
    D_hid = Wh.shape[0]
    hs = np.zeros((n, T + 1, D_hid))
    for t in range(T):
        hs[:, t+1] = np.tanh(imgs[:, t] @ Wx + hs[:, t] @ Wh + bh)
    return hs

def rnn_backward(dh_last, imgs, hs, Wx, Wh):
    n, T, D_in = imgs.shape
    D_hid = Wh.shape[0]
    dWx = np.zeros_like(Wx); dWh = np.zeros_like(Wh); dbh = np.zeros(D_hid)
    dh_next = dh_last.copy()
    for t in reversed(range(T)):
        dtanh = dh_next * (1 - hs[:, t+1]**2)
        dWx += imgs[:, t].T @ dtanh
        dWh += hs[:, t].T @ dtanh
        dbh += dtanh.sum(0)
        dh_next = dtanh @ Wh.T
    return dWx, dWh, dbh

r3 = np.random.default_rng(33)
D_in, D_hid = 8, 16
Wx = r3.normal(0, np.sqrt(1/D_in), (D_in, D_hid))
Wh = r3.normal(0, np.sqrt(1/D_hid), (D_hid, D_hid))
bh = np.zeros(D_hid)

imgs = Xtr_img[:4]
hs = rnn_forward(imgs, Wx, Wh, bh)
dh_last = r3.normal(size=(4, D_hid))           # a stand-in upstream gradient
dWx, dWh, dbh = rnn_backward(dh_last, imgs, hs, Wx, Wh)

def loss_fn(Wx_, Wh_, bh_):
    hs_ = rnn_forward(imgs, Wx_, Wh_, bh_)
    return np.sum(hs_[:, -1] * dh_last)

eps = 1e-5
print(f"{'target':>14}{'analytic':>12}{'numerical':>12}{'match':>8}")
for (i, j) in [(4, 8), (3, 4)]:
    orig = Wx[i, j]
    Wx[i, j] = orig + eps; lp = loss_fn(Wx, Wh, bh)
    Wx[i, j] = orig - eps; lm = loss_fn(Wx, Wh, bh)
    Wx[i, j] = orig
    numeric = (lp - lm) / (2 * eps)
    print(f"Wx{(i,j)}{dWx[i,j]:>12.6f}{numeric:>12.6f}"
          f"{str(abs(numeric-dWx[i,j])<1e-4):>8}")
for (i, j) in [(12, 8), (4, 8)]:
    orig = Wh[i, j]
    Wh[i, j] = orig + eps; lp = loss_fn(Wx, Wh, bh)
    Wh[i, j] = orig - eps; lm = loss_fn(Wx, Wh, bh)
    Wh[i, j] = orig
    numeric = (lp - lm) / (2 * eps)
    print(f"Wh{(i,j)}{dWh[i,j]:>12.6f}{numeric:>12.6f}"
          f"{str(abs(numeric-dWh[i,j])<1e-4):>8}")

        target    analytic   numerical   match
Wx(4, 8)    4.875486    4.875486    True
Wx(3, 4)    3.821714    3.821714    True
Wh(12, 8)   -5.155417   -5.155417    True
Wh(4, 8)   -4.609159   -4.609159    True


### Block 4  (`c4.py`)

In [6]:
# Train an RNN reading each digit row by row, and compare honestly
# against Chapter 30's fully connected network and Chapter 32's CNN,
# all on the identical images.
r4 = np.random.default_rng(33)
D_in, D_hid, D_out = 8, 16, 10
Wx = r4.normal(0, np.sqrt(1/D_in), (D_in, D_hid))
Wh = r4.normal(0, np.sqrt(1/D_hid), (D_hid, D_hid))
bh = np.zeros(D_hid)
Wo = r4.normal(0, np.sqrt(1/D_hid), (D_hid, D_out))
bo = np.zeros(D_out)

Ytr = np.eye(10)[ytr]
eta = 0.5

print(f"{'epoch':>7}{'train loss':>13}{'test accuracy':>15}")
for epoch in range(151):
    hs = rnn_forward(Xtr_img, Wx, Wh, bh)
    h_last = hs[:, -1]
    p = softmax(h_last @ Wo + bo)
    loss = -np.sum(Ytr * np.log(p + 1e-12)) / len(Xtr_img)

    dscore = (p - Ytr) / len(Xtr_img)
    dWo = h_last.T @ dscore
    dbo = dscore.sum(0)
    dh_last = dscore @ Wo.T
    dWx, dWh, dbh = rnn_backward(dh_last, Xtr_img, hs, Wx, Wh)

    Wo -= eta * dWo; bo -= eta * dbo
    Wx -= eta * dWx; Wh -= eta * dWh; bh -= eta * dbh

    if epoch % 30 == 0:
        hs_te = rnn_forward(Xte_img, Wx, Wh, bh)
        pred = softmax(hs_te[:, -1] @ Wo + bo).argmax(1)
        acc = (pred == yte).mean()
        print(f"{epoch:>7}{loss:>13.4f}{acc:>15.4f}")

hs_te = rnn_forward(Xte_img, Wx, Wh, bh)
rnn_pred = softmax(hs_te[:, -1] @ Wo + bo).argmax(1)
rnn_acc = (rnn_pred == yte).mean()
print(f"\nfinal RNN test accuracy: {rnn_acc:.4f}")

baseline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
baseline.fit(Xtr, ytr)
print(f"fully connected baseline (Chapter 30 style): "
      f"{baseline.score(Xte, yte):.4f}")

  epoch   train loss  test accuracy
      0       2.4299         0.1861
     30       1.0967         0.7028
     60       0.6307         0.7861
     90       0.4359         0.8722


    120       0.2831         0.9056


    150       0.2636         0.8194

final RNN test accuracy: 0.8194
fully connected baseline (Chapter 30 style): 0.9667


### Block 5  (`c5.py`)

In [7]:
# Step 2 showed the gradient collapsing along the time axis in a small
# network of its own. Here the same mechanism is watched inside a
# network being TRAINED: a task whose answer depends only on the FIRST
# element, so what grows is not the answer but the steps behind it.
def make_recall_task(n_samples, seq_len, seed):
    rr = np.random.default_rng(seed)
    first = rr.integers(0, 3, n_samples)               # the answer to recall
    seq = rr.normal(0, 0.3, (n_samples, seq_len, 3))
    seq[np.arange(n_samples), 0] = np.eye(3)[first]      # plant it at step 0
    return seq, first

def train_recall(seq_len, seed, epochs=120, D_hid=12):
    Xs, ys = make_recall_task(600, seq_len, seed)
    Xs_te, ys_te = make_recall_task(200, seq_len, seed + 1)
    rr = np.random.default_rng(seed)
    D_in, D_out = 3, 3
    Wx = rr.normal(0, np.sqrt(1/D_in), (D_in, D_hid))
    Wh = rr.normal(0, np.sqrt(1/D_hid), (D_hid, D_hid))
    bh = np.zeros(D_hid)
    Wo = rr.normal(0, np.sqrt(1/D_hid), (D_hid, D_out))
    bo = np.zeros(D_out)
    Y = np.eye(3)[ys]
    eta = 0.5
    for _ in range(epochs):
        hs = rnn_forward(Xs, Wx, Wh, bh)
        h_last = hs[:, -1]
        p = softmax(h_last @ Wo + bo)
        dscore = (p - Y) / len(Xs)
        dWo = h_last.T @ dscore; dbo = dscore.sum(0)
        dh_last = dscore @ Wo.T
        dWx, dWh, dbh = rnn_backward(dh_last, Xs, hs, Wx, Wh)
        Wo -= eta*dWo; bo -= eta*dbo; Wx -= eta*dWx; Wh -= eta*dWh
        bh -= eta*dbh
    hs_te = rnn_forward(Xs_te, Wx, Wh, bh)
    acc = (softmax(hs_te[:, -1] @ Wo + bo).argmax(1) == ys_te).mean()
    return acc

plain_recall = {}                      # carried forward to Step 6
print(f"{'sequence length':>16}{'recall accuracy':>18}")
for L in (2, 5, 10, 20, 40):
    acc = train_recall(L, seed=33)
    plain_recall[L] = acc
    print(f"{L:>16}{acc:>18.4f}")
print(f"\nchance accuracy on three classes: 0.333")

 sequence length   recall accuracy
               2            1.0000
               5            1.0000


              10            0.6600


              20            0.3050


              40            0.3250

chance accuracy on three classes: 0.333


### Block 6  (`c6.py`)

In [8]:
# Attention removes the bottleneck by letting the output look at EVERY
# hidden state, not just the last, weighted by how relevant each one is.
# A learned query vector scores every timestep; softmax turns the
# scores into weights; the output uses their weighted sum.
def train_recall_attention(seq_len, seed, epochs=120, D_hid=12):
    Xs, ys = make_recall_task(600, seq_len, seed)
    Xs_te, ys_te = make_recall_task(200, seq_len, seed + 1)
    rr = np.random.default_rng(seed)
    D_in, D_out = 3, 3
    Wx = rr.normal(0, np.sqrt(1/D_in), (D_in, D_hid))
    Wh = rr.normal(0, np.sqrt(1/D_hid), (D_hid, D_hid))
    bh = np.zeros(D_hid)
    q = rr.normal(0, np.sqrt(1/D_hid), D_hid)             # the learned query
    Wo = rr.normal(0, np.sqrt(1/D_hid), (D_hid, D_out))
    bo = np.zeros(D_out)
    Y = np.eye(3)[ys]
    eta = 0.5
    for _ in range(epochs):
        hs = rnn_forward(Xs, Wx, Wh, bh)[:, 1:]     # drop the t=0 zero state
        scores = hs @ q                             # (n, T)
        weights = softmax(scores)                   # attention weights
        # weighted sum of states
        context = np.einsum('nt,nth->nh', weights, hs)
        p = softmax(context @ Wo + bo)
        dscore = (p - Y) / len(Xs)
        dWo = context.T @ dscore; dbo = dscore.sum(0)
        dcontext = dscore @ Wo.T
        dweights = np.einsum('nh,nth->nt', dcontext, hs)
        dscores = weights * (dweights - (dweights * weights).sum(1,
                             keepdims=True))
        dhs = np.einsum('nt,nh->nth', weights,
                        dcontext) + np.einsum('nt,h->nth', dscores, q)
        dq = np.einsum('nth,nt->h', hs, dscores)
        # Attention gives every step its own gradient, so all of
        # dhs enters the walk back, not only the last step's.
        # Feeding only dhs[:, -1] to Step 3's routine would
        # discard the rest and leave the recurrent weights
        # on the same single long path this step removes.
        hs_full = np.concatenate(
            [np.zeros((len(Xs), 1, D_hid)), hs], axis=1)
        dWx = np.zeros_like(Wx); dWh = np.zeros_like(Wh)
        dbh = np.zeros(D_hid); dh_next = np.zeros((len(Xs), D_hid))
        for t in reversed(range(seq_len)):
            dh = dh_next + dhs[:, t]
            dtanh = dh * (1 - hs_full[:, t+1]**2)
            dWx += Xs[:, t].T @ dtanh
            dWh += hs_full[:, t].T @ dtanh
            dbh += dtanh.sum(0)
            dh_next = dtanh @ Wh.T
        Wo -= eta*dWo; bo -= eta*dbo; q -= eta*dq
        Wx -= eta*dWx; Wh -= eta*dWh; bh -= eta*dbh
    hs_te = rnn_forward(Xs_te, Wx, Wh, bh)[:, 1:]
    w_te = softmax(hs_te @ q)
    ctx_te = np.einsum('nt,nth->nh', w_te, hs_te)
    acc = (softmax(ctx_te @ Wo + bo).argmax(1) == ys_te).mean()
    return acc, w_te

print(f"{'sequence length':>16}{'plain RNN':>12}{'with attention':>16}")
for L in (2, 5, 10, 20, 40):
    acc_attn, w = train_recall_attention(L, seed=33)
    acc_plain = plain_recall[L]           # the value Step 5 just computed
    print(f"{L:>16}{acc_plain:>12.4f}{acc_attn:>16.4f}")

print(f"\nattention weights on the length-40 task, averaged "
      f"over the test set:")
print(w.mean(0).round(3))

 sequence length   plain RNN  with attention
               2      1.0000          1.0000
               5      1.0000          1.0000


              10      0.6600          1.0000


              20      0.3050          1.0000


              40      0.3250          0.9950

attention weights on the length-40 task, averaged over the test set:
[0.084 0.036 0.026 0.025 0.024 0.025 0.024 0.024 0.023 0.023 0.023 0.023
 0.023 0.023 0.023 0.023 0.023 0.023 0.023 0.023 0.023 0.023 0.023 0.023
 0.023 0.023 0.023 0.023 0.023 0.023 0.023 0.022 0.023 0.023 0.023 0.023
 0.023 0.022 0.023 0.022]
